## Setup & Imports

In [1]:
# Install packages
!pip install -q pandas numpy scipy scikit-learn xgboost matplotlib seaborn plotly tqdm joblib ipywidgets

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import poisson
from scipy.optimize import minimize
from scipy.special import softmax

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import IsotonicRegression
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅ All packages ready!")

✅ All packages ready!


## ============================================================
## CELL 3: LOAD DATA
## ============================================================



In [2]:
# PHASE 1: DATA & FEATURE HARDENING

from datetime import datetime
import pandas as pd

LEAGUES = {
    "Premier League": "E0",
    "La Liga": "SP1", 
    "Serie A": "I1",
    "Bundesliga": "D1",
    "Ligue 1": "F1"
}


def get_last_n_seasons(n=5):
    """Get last N seasons dynamically."""
    
    current_year = datetime.now().year
    
    if datetime.now().month < 8:
        current_year -= 1
    
    seasons = []
    
    for i in range(n):
        year = current_year - i
        seasons.append(f"{str(year)[-2:]}{str(year+1)[-2:]}")
    
    return seasons[::-1]


# ===============================
# STAGE 1 — DATA VALIDATION
# ===============================

def validate_dataset(df):

    print("\n==============================")
    print("STAGE 1 — DATA INTEGRITY CHECK")
    print("==============================")

    print("\nMissing values per column:")
    print(df.isna().sum())

    duplicates = df.duplicated().sum()
    print(f"\nDuplicate rows: {duplicates}")

    match_duplicates = df.duplicated(
        subset=["Date","HomeTeam","AwayTeam","League"]
    ).sum()

    print(f"Duplicate matches: {match_duplicates}")

    print("\nMatches per league:")
    print(df["League"].value_counts())

    print("\nUnique teams:")
    print(df[["HomeTeam","AwayTeam"]].nunique())

    print("\nDate range:")
    print(df["Date"].min(), "→", df["Date"].max())

    future_matches = (df["Date"] > datetime.now()).sum()

    if future_matches > 0:
        print(f"⚠️ WARNING: {future_matches} matches have future dates")

    print("\nValidation complete.")
    print("==============================\n")


# ===============================
# STAGE 3 — ELO TEAM STRENGTH
# ===============================

def compute_elo_ratings(df, k=20, base_rating=1500):

    df = df.sort_values(["League","Date"]).reset_index(drop=True)

    ratings = {}

    home_elo = []
    away_elo = []

    for _, row in df.iterrows():

        home = row["HomeTeam"]
        away = row["AwayTeam"]

        if home not in ratings:
            ratings[home] = base_rating

        if away not in ratings:
            ratings[away] = base_rating

        r_home = ratings[home]
        r_away = ratings[away]

        home_elo.append(r_home)
        away_elo.append(r_away)

        expected_home = 1 / (1 + 10 ** ((r_away - r_home) / 400))
        expected_away = 1 - expected_home

        if row["FTR"] == "H":
            s_home, s_away = 1, 0
        elif row["FTR"] == "A":
            s_home, s_away = 0, 1
        else:
            s_home, s_away = 0.5, 0.5

        ratings[home] += k * (s_home - expected_home)
        ratings[away] += k * (s_away - expected_away)

    df["ELO_home"] = home_elo
    df["ELO_away"] = away_elo
    df["ELO_diff"] = df["ELO_home"] - df["ELO_away"]

    print("✅ ELO ratings computed")

    return df


# ===============================
# DATA LOADER
# ===============================

def load_hardened_data():

    seasons = get_last_n_seasons(5)

    print(f"📊 Loading seasons: {seasons}")

    all_data = []

    for league_name, league_code in LEAGUES.items():

        for season in seasons:

            url = f"https://www.football-data.co.uk/mmz4281/{season}/{league_code}.csv"

            try:

                df = pd.read_csv(url, encoding="latin1", on_bad_lines="skip")

                df["League"] = league_name
                df["Season"] = season

                all_data.append(df)

            except Exception:
                print(f"⚠️ Failed loading {league_name} {season}")

    df = pd.concat(all_data, ignore_index=True)

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

    df = df.sort_values(["League","Date"]).reset_index(drop=True)

    required = ["Date","HomeTeam","AwayTeam","FTHG","FTAG","FTR"]

    df = df.dropna(subset=required)

    df["FTHG"] = df["FTHG"].astype(int)
    df["FTAG"] = df["FTAG"].astype(int)

    df["Outcome"] = df["FTR"].map({"H":0,"D":1,"A":2})

    df["DaysSinceMatch"] = (df["Date"].max() - df["Date"]).dt.days

    # =========================
    # STAGE 1 VALIDATION
    # =========================

    validate_dataset(df)

    # =========================
    # STAGE 3 ELO RATINGS
    # =========================

    df = compute_elo_ratings(df)

    print(f"✅ Loaded {len(df):,} matches")

    return df


# ===============================
# LOAD DATA
# ===============================

df_raw = load_hardened_data()

📊 Loading seasons: ['2122', '2223', '2324', '2425', '2526']
⚠️ Failed loading Ligue 1 2122

STAGE 1 — DATA INTEGRITY CHECK

Missing values per column:
ï»¿Div            4644
Date                 0
Time                 0
HomeTeam             0
AwayTeam             0
                  ... 
LBCH              7020
LBCD              7020
LBCA              7020
Outcome              0
DaysSinceMatch       0
Length: 167, dtype: int64

Duplicate rows: 0
Duplicate matches: 0

Matches per league:
League
Premier League    1811
Serie A           1790
La Liga           1780
Bundesliga        1440
Ligue 1           1208
Name: count, dtype: int64

Unique teams:
HomeTeam    129
AwayTeam    129
dtype: int64

Date range:
2021-08-13 00:00:00 → 2026-03-05 00:00:00

Validation complete.

✅ ELO ratings computed
✅ Loaded 8,029 matches


## ============================================================
## CELL 4: FEATURE ENGINEERING
## ============================================================


In [3]:
def create_features(df):
    """Zero-leakage feature engineering with advanced football metrics."""
    
    df = df.copy().sort_values(['League', 'Date']).reset_index(drop=True)
    
    print("🔧 Creating features...")

    # Ensure required columns exist (some leagues/seasons miss them)
    for col in ['HS','AS','HST','AST','HC','AC']:
        if col not in df.columns:
            df[col] = 0

    # =============================
    # GOALS FEATURES
    # =============================

    for window in [5, 10]:

        df[f'HGS_L{window}'] = df.groupby(['League','HomeTeam'])['FTHG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )

        df[f'HGC_L{window}'] = df.groupby(['League','HomeTeam'])['FTAG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )

        df[f'AGS_L{window}'] = df.groupby(['League','AwayTeam'])['FTAG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )

        df[f'AGC_L{window}'] = df.groupby(['League','AwayTeam'])['FTHG'].transform(
            lambda x: x.shift(1).rolling(window, min_periods=1).mean()
        )

    # =============================
    # SHOT FEATURES (xG proxy)
    # =============================

    df['HS_L5'] = df.groupby(['League','HomeTeam'])['HS'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    df['AS_L5'] = df.groupby(['League','AwayTeam'])['AS'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    df['HST_L5'] = df.groupby(['League','HomeTeam'])['HST'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    df['AST_L5'] = df.groupby(['League','AwayTeam'])['AST'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    # =============================
    # CORNER PRESSURE
    # =============================

    df['HC_L5'] = df.groupby(['League','HomeTeam'])['HC'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    df['AC_L5'] = df.groupby(['League','AwayTeam'])['AC'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).mean()
    )

    # =============================
    # TEAM FORM
    # =============================

    df['HP'] = (df['FTR'] == 'H')*3 + (df['FTR'] == 'D')*1
    df['AP'] = (df['FTR'] == 'A')*3 + (df['FTR'] == 'D')*1

    df['HForm'] = df.groupby(['League','HomeTeam'])['HP'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )

    df['AForm'] = df.groupby(['League','AwayTeam'])['AP'].transform(
        lambda x: x.shift(1).rolling(5, min_periods=1).sum()
    )

    # =============================
    # REST DAYS (fatigue)
    # =============================

    df['HomeRest'] = df.groupby(['League','HomeTeam'])['Date'].diff().dt.days
    df['AwayRest'] = df.groupby(['League','AwayTeam'])['Date'].diff().dt.days
    df['RestDiff'] = df['HomeRest'] - df['AwayRest']

    # =============================
    # MATCHUP DIFFERENCES (very predictive)
    # =============================

    df['AttackDiff'] = df['HGS_L5'] - df['AGC_L5']
    df['DefenseDiff'] = df['HGC_L5'] - df['AGS_L5']
    df['ShotDiff'] = df['HS_L5'] - df['AS_L5']
    df['ShotTargetDiff'] = df['HST_L5'] - df['AST_L5']
    df['CornerDiff'] = df['HC_L5'] - df['AC_L5']

    # =============================
    # ELO FEATURES (Stage 3)
    # =============================

    df['ELO_diff'] = df['ELO_home'] - df['ELO_away']

    # =============================
    # LEAGUE DUMMIES
    # =============================

    league_dummies = pd.get_dummies(df['League'], prefix='Lg')
    df = pd.concat([df, league_dummies], axis=1)

    # =============================
    # FEATURE LIST
    # =============================

    feature_cols = [
        c for c in df.columns
        if (
            '_L' in c
            or 'Form' in c
            or 'Rest' in c
            or 'Diff' in c
            or 'ELO' in c
            or 'Lg_' in c
        )
    ]

    # Remove rows without enough history
    df = df.dropna(subset=feature_cols)

    print(f"✅ {len(feature_cols)} features created")
    print(f"📊 {len(df):,} matches after feature engineering")

    return df, feature_cols


df, feature_cols = create_features(df_raw)

🔧 Creating features...
✅ 32 features created
📊 7,862 matches after feature engineering


## Cell 5:  WALK-FORWARD TRAINING


In [4]:
# ===============================
# WALK-FORWARD TRAINING (TRUE DEPLOYMENT STYLE)
# ===============================

def walk_forward_train(df, feature_cols):

    print("🎯 Walk-Forward Retraining by Season...")

    df = df.sort_values("Date").reset_index(drop=True)

    seasons = sorted(df["Season"].unique())

    oof_preds = np.zeros((len(df), 3))
    metrics = []

    for i in range(2, len(seasons)):

        train_seasons = seasons[:i]
        test_season = seasons[i]

        train_idx = df["Season"].isin(train_seasons)
        test_idx = df["Season"] == test_season

        train = df.loc[train_idx]
        test = df.loc[test_idx]

        X_train = train[feature_cols].fillna(0).values
        y_train = train["Outcome"].values

        X_test = test[feature_cols].fillna(0).values
        y_test = test["Outcome"].values

        print(f"\n📅 Train: {train_seasons[0]} → {train_seasons[-1]} | Test: {test_season}")

        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            max_depth=6,
            learning_rate=0.03,
            n_estimators=400,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbosity=0
        )

        model.fit(X_train, y_train)

        preds = model.predict_proba(X_test)

        oof_preds[test.index] = preds

        # Log-loss
        ll = log_loss(y_test, preds)

        # Brier score
        y_test_onehot = np.zeros((len(y_test), 3))
        y_test_onehot[np.arange(len(y_test)), y_test] = 1

        brier = np.mean(np.sum((preds - y_test_onehot) ** 2, axis=1))

        print(f"   Log-Loss={ll:.4f} | Brier={brier:.4f}")

        metrics.append({
            "season": test_season,
            "log_loss": ll,
            "brier": brier
        })

    print("\n📊 WALK-FORWARD PERFORMANCE")

    mean_ll = np.mean([m["log_loss"] for m in metrics])
    mean_brier = np.mean([m["brier"] for m in metrics])

    print(f"✅ Mean Log-Loss: {mean_ll:.4f}")
    print(f"✅ Mean Brier Score: {mean_brier:.4f}")

    return oof_preds, metrics


# Run training
oof_preds, metrics = walk_forward_train(df, feature_cols)

🎯 Walk-Forward Retraining by Season...

📅 Train: 2122 → 2223 | Test: 2324
   Log-Loss=1.0348 | Brier=0.6139

📅 Train: 2122 → 2324 | Test: 2425
   Log-Loss=1.0411 | Brier=0.6210

📅 Train: 2122 → 2425 | Test: 2526
   Log-Loss=1.0127 | Brier=0.6024

📊 WALK-FORWARD PERFORMANCE
✅ Mean Log-Loss: 1.0295
✅ Mean Brier Score: 0.6124


## PROBABILITY CALIBRATION

In [5]:
# PROBABILITY CALIBRATION

class ProbabilityCalibrator:
    """Isotonic regression calibration for multi-class probabilities."""
    
    def __init__(self):
        self.calibrators = []
    
    def fit(self, y_true, y_pred_proba):

        self.calibrators = []

        for i in range(3):

            cal = IsotonicRegression(out_of_bounds='clip')

            y_binary = (y_true == i).astype(int)

            cal.fit(y_pred_proba[:, i], y_binary)

            self.calibrators.append(cal)
    
    def transform(self, y_pred_proba):

        calibrated = np.zeros_like(y_pred_proba)

        for i, cal in enumerate(self.calibrators):

            calibrated[:, i] = cal.transform(y_pred_proba[:, i])

        # Normalize probabilities
        calibrated_sum = calibrated.sum(axis=1, keepdims=True)

        calibrated = calibrated / np.clip(calibrated_sum, 1e-8, None)

        return calibrated


# =============================
# FIT CALIBRATOR
# =============================

calibrator = ProbabilityCalibrator()

calibrator.fit(df['Outcome'].values, oof_preds)

calibrated_preds = calibrator.transform(oof_preds)


# =============================
# EVALUATE CALIBRATION
# =============================

raw_ll = log_loss(df['Outcome'].values, oof_preds)
cal_ll = log_loss(df['Outcome'].values, calibrated_preds)

print("\n📊 Calibration Results")
print(f"   Raw Log-Loss: {raw_ll:.4f}")
print(f"   Calibrated Log-Loss: {cal_ll:.4f}")
print(f"   Improvement: {(raw_ll - cal_ll):.4f}")


# =============================
# ADD PROBABILITIES TO DATA
# =============================

df['prob_home'] = calibrated_preds[:, 0]
df['prob_draw'] = calibrated_preds[:, 1]
df['prob_away'] = calibrated_preds[:, 2]


📊 Calibration Results
   Raw Log-Loss: 15.1724
   Calibrated Log-Loss: 1.0715
   Improvement: 14.1009


## ============================================================
## CELL 5: DIXON-COLES MODEL
## ============================================================


In [6]:
# ===============================
# DIXON-COLES (EXACT + TIME DECAY + TUNING)
# ===============================

class DixonColesTimeDecay:
    
    def __init__(self, xi=0.002, max_goals=10):
        self.xi = xi
        self.max_goals = max_goals
        self.teams = None
        self.attack = None
        self.defence = None
        self.home_adv = None
        self.rho = None


    # ===============================
    # MODEL FIT
    # ===============================

    def fit(self, df, league=None):

        if league:
            data = df[df["League"] == league].copy()
        else:
            data = df.copy()

        data = data.sort_values("Date")

        self.teams = sorted(set(data["HomeTeam"]) | set(data["AwayTeam"]))
        n = len(self.teams)

        team_to_idx = {team: i for i, team in enumerate(self.teams)}

        home_idx = data["HomeTeam"].map(team_to_idx).values
        away_idx = data["AwayTeam"].map(team_to_idx).values

        hg = data["FTHG"].values
        ag = data["FTAG"].values

        days = data["DaysSinceMatch"].values
        weights = np.exp(-self.xi * days)


        def nll(params):

            att = params[:n]
            deff = params[n:2*n]
            home = params[2*n]
            rho = params[2*n+1]

            # Normalize attack parameters (prevents drift)
            att = att - np.mean(att)

            lh = np.exp(home + att[home_idx] - deff[away_idx])
            la = np.exp(att[away_idx] - deff[home_idx])

            p = poisson.pmf(hg, lh) * poisson.pmf(ag, la)

            corr = np.ones_like(p)

            mask00 = (hg == 0) & (ag == 0)
            mask01 = (hg == 0) & (ag == 1)
            mask10 = (hg == 1) & (ag == 0)
            mask11 = (hg == 1) & (ag == 1)

            corr[mask00] = 1 - lh[mask00] * la[mask00] * rho
            corr[mask01] = 1 + lh[mask01] * rho
            corr[mask10] = 1 + la[mask10] * rho
            corr[mask11] = 1 - rho

            p *= corr

            ll = np.sum(weights * np.log(np.maximum(p, 1e-12)))

            return -ll


        x0 = np.concatenate([np.zeros(n), np.zeros(n), [0.2], [0]])

        bounds = [(-3,3)]*(2*n) + [(-1,1),(-0.1,0.1)]

        res = minimize(
            nll,
            x0,
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter":200}
        )

        params = res.x

        att = params[:n] - np.mean(params[:n])
        deff = params[n:2*n]

        self.attack = dict(zip(self.teams, att))
        self.defence = dict(zip(self.teams, deff))
        self.home_adv = params[2*n]
        self.rho = params[2*n+1]

        return self


    # ===============================
    # EXACT PROBABILITY PREDICTION
    # ===============================

    def predict(self, home, away):

        lh = np.exp(self.home_adv + self.attack[home] - self.defence[away])
        la = np.exp(self.attack[away] - self.defence[home])

        max_g = self.max_goals

        home_probs = poisson.pmf(range(max_g+1), lh)
        away_probs = poisson.pmf(range(max_g+1), la)

        score_matrix = np.outer(home_probs, away_probs)

        # Dixon-Coles low score correction
        score_matrix[0,0] *= (1 - lh*la*self.rho)
        score_matrix[0,1] *= (1 + lh*self.rho)
        score_matrix[1,0] *= (1 + la*self.rho)
        score_matrix[1,1] *= (1 - self.rho)

        prob_home = np.tril(score_matrix, -1).sum()
        prob_draw = np.trace(score_matrix)
        prob_away = np.triu(score_matrix, 1).sum()

        total_goals = np.add.outer(range(max_g+1), range(max_g+1))

        prob_over25 = score_matrix[total_goals > 2].sum()
        prob_under25 = 1 - prob_over25

        return {
            "lambda_home": float(lh),
            "lambda_away": float(la),
            "prob_home": float(prob_home),
            "prob_draw": float(prob_draw),
            "prob_away": float(prob_away),
            "prob_over_25": float(prob_over25),
            "prob_under_25": float(prob_under25),
            "exp_goals": float(lh + la)
        }


# ===============================
# TUNE TIME DECAY PARAMETER
# ===============================

def tune_dixon_coles_xi(df, league=None):

    xis = [0.001, 0.002, 0.003]

    best_xi = None
    best_ll = np.inf

    print("🔍 Tuning Dixon-Coles xi parameter...")

    for xi in xis:

        model = DixonColesTimeDecay(xi=xi)
        model.fit(df, league)

        # crude evaluation using home advantage magnitude
        score = abs(model.home_adv)

        if score < best_ll:
            best_ll = score
            best_xi = xi

    print("✅ Best xi:", best_xi)

    return best_xi

## ============================================================
## CELL 6:  LOG-ODDS ENSEMBLE BLENDING
## ============================================================


In [7]:
def ensemble_predict_fixed(league, home, away, features, dc_weight=0.6):
    """
    Combine ML model + Dixon-Coles probabilities using log pooling.
    """

    # -------------------------
    # Dixon-Coles prediction
    # -------------------------
    
    dc_pred = dc_models[league].predict(home, away)

    dc_probs = np.array([
        dc_pred["prob_home"],
        dc_pred["prob_draw"],
        dc_pred["prob_away"]
    ])


    # -------------------------
    # ML prediction
    # -------------------------
    
    ml_probs = final_model.predict_proba(features.reshape(1, -1))[0]


    # -------------------------
    # Safety clipping
    # -------------------------
    
    dc_probs = np.clip(dc_probs, 1e-9, 1 - 1e-9)
    ml_probs = np.clip(ml_probs, 1e-9, 1 - 1e-9)


    # -------------------------
    # Log probability pooling
    # -------------------------
    
    dc_log = np.log(dc_probs)
    ml_log = np.log(ml_probs)

    combined_log = dc_weight * dc_log + (1 - dc_weight) * ml_log

    ensemble = softmax(combined_log)


    # -------------------------
    # Return results
    # -------------------------
    
    return {
        "prob_home": float(ensemble[0]),
        "prob_draw": float(ensemble[1]),
        "prob_away": float(ensemble[2]),

        "lambda_home": float(dc_pred["lambda_home"]),
        "lambda_away": float(dc_pred["lambda_away"]),

        "prob_over_25": float(dc_pred["prob_over_25"]),
        "prob_under_25": float(dc_pred["prob_under_25"]),

        "exp_goals": float(dc_pred["exp_goals"])
    }

## MARKET PRICING & EV CALCULATION

In [8]:
# ===============================
# PHASE 6: MARKET PRICING & EV
# ===============================

def calculate_market_edge(model_prob, bookmaker_odds):
    """
    Calculate betting edge and expected value.
    """

    if bookmaker_odds is None or pd.isna(bookmaker_odds) or bookmaker_odds <= 1.0:
        return None

    implied_prob = 1.0 / bookmaker_odds
    edge = model_prob - implied_prob
    ev = (model_prob * bookmaker_odds) - 1

    return {
        "model_prob": model_prob,
        "bookmaker_odds": bookmaker_odds,
        "implied_prob": implied_prob,
        "edge": edge,
        "edge_pct": edge * 100,
        "ev": ev,
        "ev_pct": ev * 100,
        "has_value": ev > 0.03
    }


# ===============================
# ASIAN HANDICAP PROBABILITIES
# ===============================

def asian_handicap_probabilities(lh, la, max_goals=10):

    home_probs = poisson.pmf(range(max_goals+1), lh)
    away_probs = poisson.pmf(range(max_goals+1), la)

    score_matrix = np.outer(home_probs, away_probs)

    goal_diff = np.subtract.outer(range(max_goals+1), range(max_goals+1))

    probs = {}

    # AH 0 (Draw No Bet equivalent)
    probs["ah_home_0"] = score_matrix[goal_diff > 0].sum()
    probs["ah_away_0"] = score_matrix[goal_diff < 0].sum()

    # AH -0.5
    probs["ah_home_-0.5"] = score_matrix[goal_diff > 0].sum()
    probs["ah_away_+0.5"] = score_matrix[goal_diff <= 0].sum()

    # AH -1
    probs["ah_home_-1"] = score_matrix[goal_diff > 1].sum()
    probs["ah_away_+1"] = score_matrix[goal_diff <= 1].sum()

    return probs


# ===============================
# BUILD ALL MARKET PROBABILITIES
# ===============================

def build_market_probabilities(pred):

    probs = {}

    # 1X2
    probs["home"] = pred["prob_home"]
    probs["draw"] = pred["prob_draw"]
    probs["away"] = pred["prob_away"]

    # Over / Under
    probs["over_2.5"] = pred["prob_over_25"]
    probs["under_2.5"] = pred["prob_under_25"]

    # Double Chance
    probs["1X"] = pred["prob_home"] + pred["prob_draw"]
    probs["X2"] = pred["prob_draw"] + pred["prob_away"]
    probs["12"] = pred["prob_home"] + pred["prob_away"]

    # Draw No Bet
    probs["dnb_home"] = pred["prob_home"] / (1 - pred["prob_draw"])
    probs["dnb_away"] = pred["prob_away"] / (1 - pred["prob_draw"])

    # Asian Handicap
    ah_probs = asian_handicap_probabilities(
        pred["lambda_home"],
        pred["lambda_away"]
    )

    probs.update(ah_probs)

    return probs


# ===============================
# FIND VALUE BETS
# ===============================

def find_value_bets(prediction, odds_dict, min_ev=0.03):

    market_probs = build_market_probabilities(prediction)

    value_bets = []

    for market, prob in market_probs.items():

        odds = odds_dict.get(market)

        if odds is None:
            continue

        result = calculate_market_edge(prob, odds)

        if result and result["ev"] > min_ev:
            result["market"] = market
            value_bets.append(result)

    value_bets.sort(key=lambda x: x["ev"], reverse=True)

    return value_bets


print("✅ Market pricing engine ready (1X2 + O/U + DC + DNB + Asian Handicap)")

✅ Market pricing engine ready (1X2 + O/U + DC + DNB + Asian Handicap)


## HIGH-PROBABILITY MARKETS

In [9]:
# ===============================
# PHASE 7: HIGH-PROBABILITY MARKETS
# ===============================

def calculate_high_prob_markets(prediction, threshold=0.70):
    """
    Generate conservative high-probability betting markets.
    """

    suggestions = []

    p_home = prediction["prob_home"]
    p_draw = prediction["prob_draw"]
    p_away = prediction["prob_away"]

    lh = prediction["lambda_home"]
    la = prediction["lambda_away"]

    # ------------------------------
    # DOUBLE CHANCE
    # ------------------------------

    dc_1x = p_home + p_draw
    dc_x2 = p_draw + p_away
    dc_12 = p_home + p_away

    if dc_1x > threshold:
        suggestions.append({
            "market": "Double Chance (1X)",
            "probability": dc_1x,
            "risk": "Conservative"
        })

    if dc_x2 > threshold:
        suggestions.append({
            "market": "Double Chance (X2)",
            "probability": dc_x2,
            "risk": "Conservative"
        })

    if dc_12 > threshold:
        suggestions.append({
            "market": "Double Chance (12)",
            "probability": dc_12,
            "risk": "Conservative"
        })

    # ------------------------------
    # DRAW NO BET
    # ------------------------------

    if p_home > 0.55:
        dnb_home = p_home / (1 - p_draw)

        suggestions.append({
            "market": "Draw No Bet - Home",
            "probability": dnb_home,
            "risk": "Balanced"
        })

    if p_away > 0.55:
        dnb_away = p_away / (1 - p_draw)

        suggestions.append({
            "market": "Draw No Bet - Away",
            "probability": dnb_away,
            "risk": "Balanced"
        })

    # ------------------------------
    # EXACT POISSON GOAL GRID
    # ------------------------------

    max_goals = 10

    home_probs = poisson.pmf(range(max_goals+1), lh)
    away_probs = poisson.pmf(range(max_goals+1), la)

    score_matrix = np.outer(home_probs, away_probs)

    total_goals = np.add.outer(range(max_goals+1), range(max_goals+1))

    # ------------------------------
    # UNDER 3.5
    # ------------------------------

    prob_u35 = score_matrix[total_goals <= 3].sum()

    if prob_u35 > threshold:
        suggestions.append({
            "market": "Under 3.5 Goals",
            "probability": prob_u35,
            "risk": "Conservative"
        })

    # ------------------------------
    # OVER 1.5
    # ------------------------------

    prob_o15 = score_matrix[total_goals >= 2].sum()

    if prob_o15 > threshold:
        suggestions.append({
            "market": "Over 1.5 Goals",
            "probability": prob_o15,
            "risk": "Conservative"
        })

    # ------------------------------
    # STRONG FAVORITES
    # ------------------------------

    if p_home > 0.70:
        suggestions.append({
            "market": "Strong Favorite - Home",
            "probability": p_home,
            "risk": "Balanced"
        })

    if p_away > 0.70:
        suggestions.append({
            "market": "Strong Favorite - Away",
            "probability": p_away,
            "risk": "Balanced"
        })

    # Sort suggestions
    suggestions.sort(key=lambda x: x["probability"], reverse=True)

    return suggestions


print("✅ High-probability markets engine ready")

✅ High-probability markets engine ready


## RISK MANAGEMENT ENGINE

In [10]:
# ===============================
# PHASE 8: RISK MANAGEMENT ENGINE
# ===============================

class RiskManager:
    """
    Advanced Kelly risk manager with safety constraints.

    Rules:
    - 0.25 Kelly (fractional)
    - Max 5% per bet
    - Max 20% daily exposure
    - Max 3 bets per match
    - Stop betting at 25% drawdown
    - Minimum edge requirement
    """

    def __init__(
        self,
        bankroll=10000,
        kelly_frac=0.25,
        max_bet_pct=0.05,
        max_daily_pct=0.20,
        max_bets_per_match=3,
        min_edge=0.02
    ):

        self.initial_bankroll = bankroll
        self.bankroll = bankroll
        self.peak_bankroll = bankroll

        self.kelly_frac = kelly_frac
        self.max_bet_pct = max_bet_pct
        self.max_daily_pct = max_daily_pct
        self.max_bets_per_match = max_bets_per_match
        self.min_edge = min_edge

        self.daily_exposure = 0
        self.match_bets = {}

        self.stopped = False


    # ---------------------------
    # KELLY STAKE CALCULATION
    # ---------------------------

    def kelly_stake(self, prob, odds, edge, match_id=None):

        if self.stopped:
            return 0

        if edge < self.min_edge:
            return 0

        if prob <= 0.05 or prob >= 0.95 or odds <= 1.0:
            return 0

        # Kelly formula
        b = odds - 1
        kelly = (prob * b - (1 - prob)) / b

        if kelly <= 0:
            return 0

        # fractional Kelly
        kelly *= self.kelly_frac

        # cap stake %
        kelly = min(kelly, self.max_bet_pct)

        stake = self.bankroll * kelly

        # check daily exposure
        if self.daily_exposure + stake > self.bankroll * self.max_daily_pct:
            return 0

        # check match bet limit
        if match_id is not None:

            current = self.match_bets.get(match_id, 0)

            if current >= self.max_bets_per_match:
                return 0

            self.match_bets[match_id] = current + 1

        # round stake to realistic unit
        stake = round(stake, 2)

        # update exposure
        self.daily_exposure += stake

        return stake


    # ---------------------------
    # RECORD BET RESULT
    # ---------------------------

    def record_bet(self, stake, won, odds):

        profit = stake * (odds - 1) if won else -stake

        self.bankroll += profit

        # update peak bankroll
        if self.bankroll > self.peak_bankroll:
            self.peak_bankroll = self.bankroll

        # drawdown calculation
        drawdown = (self.peak_bankroll - self.bankroll) / self.peak_bankroll

        if drawdown >= 0.25:
            self.stopped = True
            print(f"⚠️ Drawdown stop triggered at {drawdown:.1%}")

        return profit


    # ---------------------------
    # RESET DAILY LIMITS
    # ---------------------------

    def reset_daily(self):

        self.daily_exposure = 0
        self.match_bets = {}


    # ---------------------------
    # PERFORMANCE STATS
    # ---------------------------

    def get_stats(self):

        drawdown = (self.peak_bankroll - self.bankroll) / self.peak_bankroll
        roi = (self.bankroll - self.initial_bankroll) / self.initial_bankroll

        return {
            "bankroll": round(self.bankroll, 2),
            "peak_bankroll": round(self.peak_bankroll, 2),
            "daily_exposure": round(self.daily_exposure, 2),
            "drawdown_pct": round(drawdown * 100, 2),
            "roi_pct": round(roi * 100, 2),
            "stopped": self.stopped
        }


# Initialize risk manager
risk_manager = RiskManager(bankroll=10000, kelly_frac=0.25)

print("✅ Risk management engine ready")

✅ Risk management engine ready


## PROFESSIONAL BACKTESTER

In [11]:
# ===============================
# PHASE 9: PORTFOLIO BACKTEST ENGINE
# ===============================

def run_optimal_backtest(
    df,
    initial_bankroll=10000,
    min_prob=0.55,
    min_ev=0.08,
    max_odds=3.0,
):

    class ConservativeRiskManager:

        def __init__(self, bankroll):

            self.initial_bankroll = bankroll
            self.bankroll = bankroll
            self.peak_bankroll = bankroll
            self.daily_exposure = 0
            self.stopped = False
            self.match_bets = {}

        def kelly_stake(self, prob, odds, ev, match_id):

            if self.stopped:
                return 0

            # max 3 bets per match
            if self.match_bets.get(match_id,0) >= 3:
                return 0

            b = odds - 1
            kelly = (prob*b - (1-prob))/b

            if kelly <= 0:
                return 0

            kelly *= 0.15 * min(1, ev/0.10)
            kelly = min(kelly, 0.02)

            stake = self.bankroll * kelly

            if self.daily_exposure + stake > self.bankroll * 0.10:
                return 0

            self.daily_exposure += stake
            self.match_bets[match_id] = self.match_bets.get(match_id,0)+1

            return round(stake,2)

        def record_bet(self, stake, won, odds):

            profit = stake*(odds-1) if won else -stake
            self.bankroll += profit

            if self.bankroll > self.peak_bankroll:
                self.peak_bankroll = self.bankroll

            drawdown = (self.peak_bankroll - self.bankroll)/self.peak_bankroll

            if drawdown >= 0.20:
                self.stopped = True

            return profit

        def reset_daily(self):
            self.daily_exposure = 0
            self.match_bets = {}


    risk_mgr = ConservativeRiskManager(initial_bankroll)

    trades = []

    print("📊 Running portfolio backtest")

    for idx in tqdm(range(len(df))):

        row = df.iloc[idx]

        match_id = f"{row['Date']}_{row['HomeTeam']}_{row['AwayTeam']}"

        # ===============================
        # BUILD ALL MARKETS
        # ===============================

        markets = {

            # 1X2
            "home": (row["prob_home"], row.get("AvgH"), "H"),
            "draw": (row["prob_draw"], row.get("AvgD"), "D"),
            "away": (row["prob_away"], row.get("AvgA"), "A"),

            # Over / Under
            "over25": (row.get("prob_over_25"), row.get("Avg>2.5"), None),
            "under25": (row.get("prob_under_25"), row.get("Avg<2.5"), None),

        }

        for market,(prob,odds,outcome) in markets.items():

            if prob is None or pd.isna(odds):
                continue

            if odds <= 1 or odds > max_odds:
                continue

            if prob < min_prob:
                continue

            implied = 1/odds
            ev = prob*odds - 1
            edge = prob - implied

            if ev < min_ev:
                continue

            stake = risk_mgr.kelly_stake(prob,odds,ev,match_id)

            if stake <= 0:
                continue

            # ===============================
            # DETERMINE RESULT
            # ===============================

            if market == "over25":
                won = (row["FTHG"] + row["FTAG"]) > 2
            elif market == "under25":
                won = (row["FTHG"] + row["FTAG"]) <= 2
            else:
                won = row["FTR"] == outcome

            profit = risk_mgr.record_bet(stake,won,odds)

            # ===============================
            # CLV
            # ===============================

            closing_odds = None
            if market == "home":
                closing_odds = row.get("MaxH")
            elif market == "draw":
                closing_odds = row.get("MaxD")
            elif market == "away":
                closing_odds = row.get("MaxA")

            clv = None
            if closing_odds:
                clv = (closing_odds - odds)/odds

            trades.append({

                "date":row["Date"],
                "league":row.get("League","Unknown"),
                "match":f"{row['HomeTeam']} vs {row['AwayTeam']}",
                "market":market,
                "prob":prob,
                "odds":odds,
                "ev":ev,
                "edge":edge,
                "stake":stake,
                "won":won,
                "profit":profit,
                "bankroll":risk_mgr.bankroll,
                "clv":clv

            })

        # reset exposure per day
        if idx < len(df)-1 and df.iloc[idx+1]["Date"] != row["Date"]:
            risk_mgr.reset_daily()

    trades_df = pd.DataFrame(trades)

    if len(trades_df)==0:
        print("⚠️ No trades passed filters")
        return None,None

    winners = trades_df[trades_df["won"]]
    losers = trades_df[~trades_df["won"]]

    stats = {

        "initial":initial_bankroll,
        "final":risk_mgr.bankroll,
        "profit":risk_mgr.bankroll-initial_bankroll,
        "roi_pct":(risk_mgr.bankroll-initial_bankroll)/initial_bankroll*100,

        "n_trades":len(trades_df),
        "win_rate":len(winners)/len(trades_df)*100,

        "avg_odds":trades_df["odds"].mean(),
        "avg_prob":trades_df["prob"].mean()*100,
        "avg_ev":trades_df["ev"].mean()*100,

        "profit_factor":winners["profit"].sum()/abs(losers["profit"].sum()) if len(losers)>0 else None,

        "max_drawdown":
        ((risk_mgr.peak_bankroll - trades_df["bankroll"].min())
         / risk_mgr.peak_bankroll)*100,

        "avg_clv":trades_df["clv"].mean()
    }

    print("\n==============================")
    print("BACKTEST RESULTS")
    print("==============================")

    for k,v in stats.items():

        if isinstance(v,float):
            print(f"{k:15}: {v:.2f}")
        else:
            print(f"{k:15}: {v}")

    return trades_df,stats


print("\n⚡ Running portfolio backtest...\n")

trades_df,stats = run_optimal_backtest(df,10000)


⚡ Running portfolio backtest...

📊 Running portfolio backtest


100%|██████████| 7862/7862 [00:07<00:00, 1051.88it/s]

⚠️ No trades passed filters


## ============================================================================
## CRITICAL FIX: TRAIN FINAL MODELS
## ============================================================================

In [12]:
# ============================================================================
# CRITICAL FIX: TRAIN FINAL MODELS
# ADD THIS CELL BEFORE THE GUI CELL (Cell 23)
# ============================================================================

print("="*80)
print("🚀 TRAINING FINAL MODELS FOR PREDICTIONS")
print("="*80)

# ============================================================================
# 1. TRAIN FINAL ML MODEL (CALIBRATED)
# ============================================================================

print("\n🤖 Training calibrated ML model...")

from sklearn.calibration import CalibratedClassifierCV

# Prepare full dataset
X_full = df[feature_cols].fillna(0).values
y_full = df['Outcome'].values

# Base XGBoost
base_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    max_depth=5,
    learning_rate=0.05,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

# Time-aware CV for calibration
tscv = TimeSeriesSplit(n_splits=3)

# Calibrated model
final_model = CalibratedClassifierCV(
    base_model,
    method='isotonic',
    cv=tscv
)

print("   Fitting calibrated model...")
final_model.fit(X_full, y_full)

# Evaluate
y_pred = final_model.predict_proba(X_full)
ll = log_loss(y_full, y_pred)

print(f"✅ Final ML model trained")
print(f"   Log-loss: {ll:.4f}")

# ============================================================================
# 2. TRAIN DIXON-COLES MODELS (ONE PER LEAGUE)
# ============================================================================

print("\n🎯 Training Dixon-Coles models...")

dc_models = {}

for league in df['League'].unique():
    print(f"   Training {league}...")
    
    try:
        # Create and fit model
        dc_model = DixonColesTimeDecay(xi=0.002, max_goals=10)
        dc_model.fit(df, league=league)
        
        dc_models[league] = dc_model
        
        print(f"   ✅ {league}: {len(dc_model.teams)} teams, home_adv={dc_model.home_adv:.3f}")
        
    except Exception as e:
        print(f"   ⚠️  {league} failed: {e}")

print(f"\n✅ Trained {len(dc_models)} Dixon-Coles models")

# ============================================================================
# 3. VERIFY MODELS ARE READY
# ============================================================================

print("\n🔍 Verifying models...")

# Test ML model
test_features = X_full[:1]
test_pred = final_model.predict_proba(test_features)
print(f"✅ ML model works: {test_pred}")

# Test DC models
test_league = list(dc_models.keys())[0]
test_teams = list(dc_models[test_league].teams)[:2]
test_dc_pred = dc_models[test_league].predict(test_teams[0], test_teams[1])
print(f"✅ DC model works: prob_home={test_dc_pred['prob_home']:.2%}")

print("\n" + "="*80)
print("✅ ALL MODELS READY FOR PREDICTIONS!")
print("="*80)
print("\nYou can now use the GUI to predict matches.")
print("Models available:")
print(f"  • final_model: Calibrated ML")
print(f"  • dc_models: {list(dc_models.keys())}")

🚀 TRAINING FINAL MODELS FOR PREDICTIONS

🤖 Training calibrated ML model...
   Fitting calibrated model...
✅ Final ML model trained
   Log-loss: 0.8430

🎯 Training Dixon-Coles models...
   Training Bundesliga...
   ✅ Bundesliga: 25 teams, home_adv=0.193
   Training La Liga...
   ✅ La Liga: 26 teams, home_adv=0.276
   Training Ligue 1...
   ✅ Ligue 1: 24 teams, home_adv=0.223
   Training Premier League...
   ✅ Premier League: 27 teams, home_adv=0.147
   Training Serie A...
   ✅ Serie A: 27 teams, home_adv=0.123

✅ Trained 5 Dixon-Coles models

🔍 Verifying models...
✅ ML model works: [[0.7692161  0.10385595 0.12692795]]
✅ DC model works: prob_home=8.50%

✅ ALL MODELS READY FOR PREDICTIONS!

You can now use the GUI to predict matches.
Models available:
  • final_model: Calibrated ML
  • dc_models: ['Bundesliga', 'La Liga', 'Ligue 1', 'Premier League', 'Serie A']


##  INTERACTIVE GUI

In [14]:
# ============================================================================
# IMPROVED PROFESSIONAL GUI - CLEANER LAYOUT WITH SEPARATE WINDOWS
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np

manual_matches = []
predictions_results = []

# ============================================================================
# STYLED COMPONENTS
# ============================================================================

def create_styled_box(title, children, color="#2c3e50"):
    """Create a styled container box."""
    header = widgets.HTML(
        value=f'''
        <div style="background-color: {color}; padding: 15px; border-radius: 8px 8px 0 0; margin-top: 10px;">
            <h3 style="color: white; margin: 0;">{title}</h3>
        </div>
        '''
    )
    content = widgets.VBox(
        children=children,
        layout=widgets.Layout(
            border='2px solid ' + color,
            border_radius='0 0 8px 8px',
            padding='15px',
            background='#f8f9fa'
        )
    )
    return widgets.VBox([header, content])

# ============================================================================
# INPUT SECTION
# ============================================================================

paste_area = widgets.Textarea(
    value='',
    placeholder='''Paste games here (one per line):
Serie A, Lecce, Cremonese, 2.415, 3.015, 3.705
Serie A, Bologna, Verona, 1.73, 3.805, 5.73''',
    layout=widgets.Layout(width='100%', height='180px', font_family='monospace')
)

parse_button = widgets.Button(
    description='📋 Parse Games',
    button_style='primary',
    layout=widgets.Layout(width='150px', height='40px'),
    tooltip='Parse and validate pasted games'
)

clear_button = widgets.Button(
    description='🗑️ Clear',
    button_style='warning',
    layout=widgets.Layout(width='100px', height='40px')
)

# ============================================================================
# CONTROLS SECTION
# ============================================================================

min_prob_slider = widgets.FloatSlider(
    value=0.30,
    min=0.20,
    max=0.70,
    step=0.05,
    description='Min Prob:',
    readout_format='.0%',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='450px')
)

min_ev_slider = widgets.FloatSlider(
    value=0.00,
    min=0.00,
    max=0.10,
    step=0.01,
    description='Min EV:',
    readout_format='.0%',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='450px')
)

predict_button = widgets.Button(
    description='🔮 GENERATE PREDICTIONS',
    button_style='success',
    layout=widgets.Layout(width='300px', height='50px'),
    tooltip='Run model and find value bets'
)

# ============================================================================
# OUTPUT SECTIONS
# ============================================================================

status_output = widgets.Output()
top_bets_output = widgets.Output()
all_results_output = widgets.Output()

# ============================================================================
# FUNCTIONS
# ============================================================================

def parse_games(b):
    """Parse pasted games."""
    global manual_matches
    manual_matches = []
    
    with status_output:
        clear_output()
        
        text = paste_area.value.strip()
        if not text:
            print("❌ No games pasted!")
            return
        
        lines = text.split('\n')
        parsed = 0
        errors = []
        
        for line in lines:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            try:
                parts = [p.strip() for p in line.split(',')]
                if len(parts) != 6:
                    errors.append(f"Wrong format: {line[:50]}")
                    continue
                
                manual_matches.append({
                    'league': parts[0],
                    'home': parts[1],
                    'away': parts[2],
                    'odds_home': float(parts[3]),
                    'odds_draw': float(parts[4]),
                    'odds_away': float(parts[5])
                })
                parsed += 1
                
            except Exception as e:
                errors.append(f"Error: {line[:30]}...")
        
        print(f"✅ Parsed {parsed} matches")
        
        if errors and len(errors) <= 3:
            print(f"\n⚠️  Errors:")
            for err in errors:
                print(f"   {err}")
        
        if parsed > 0:
            print(f"\n📋 Ready to predict:")
            for m in manual_matches:
                print(f"   • {m['home']} vs {m['away']} ({m['league']})")

def clear_paste(b):
    """Clear paste area."""
    paste_area.value = ''
    global manual_matches
    manual_matches = []
    
    with status_output:
        clear_output()
        print("🗑️ Cleared all")
    
    with top_bets_output:
        clear_output()
    
    with all_results_output:
        clear_output()

def predict_all(b):
    """Generate predictions."""
    global predictions_results
    predictions_results = []
    
    # Clear outputs
    with status_output:
        clear_output()
    with top_bets_output:
        clear_output()
    with all_results_output:
        clear_output()
    
    if not manual_matches:
        with status_output:
            print("❌ No matches! Parse games first.")
        return
    
    with status_output:
        print(f"🔄 Processing {len(manual_matches)} matches...")
    
    min_prob = min_prob_slider.value
    min_ev = min_ev_slider.value
    
    # Process matches
    for match in manual_matches:
        league = match['league']
        home = match['home']
        away = match['away']
        odds = np.array([match['odds_home'], match['odds_draw'], match['odds_away']])
        
        try:
            # Get features
            team_data = df[(df['League'] == league) & 
                          ((df['HomeTeam'] == home) | (df['AwayTeam'] == away))].tail(10)
            
            if len(team_data) < 3:
                team_data = df[df['League'] == league].tail(50)
            
            features = team_data[feature_cols].mean().values
            
            # Get prediction
            pred = ensemble_predict_fixed(league, home, away, features)
            
            # Calculate all markets
            league_df = df[df['League'] == league]
            row = league_df.tail(1).iloc[0]
            markets = generate_markets(pred, odds, row)
            
            # Find value bets
            bets = []
            for name, prob, market_odds in markets:
                if market_odds is None or pd.isna(market_odds) or market_odds <= 1.0:
                    continue
                
                implied = 1 / market_odds
                ev = prob * market_odds - 1
                edge = prob - implied
                
                if prob > min_prob and ev > min_ev:
                    bets.append({
                        'market': name,
                        'prob': prob,
                        'odds': market_odds,
                        'ev': ev,
                        'edge': edge
                    })
            
            bets = sorted(bets, key=lambda x: x['ev'], reverse=True)
            
            predictions_results.append({
                'match': f"{home} vs {away}",
                'league': league,
                'home': home,
                'away': away,
                'prob_home': pred['prob_home'],
                'prob_draw': pred['prob_draw'],
                'prob_away': pred['prob_away'],
                'xg': pred['exp_goals'],
                'lambda_home': pred['lambda_home'],
                'lambda_away': pred['lambda_away'],
                'bets': bets,
                'has_value': len(bets) > 0
            })
            
        except Exception as e:
            with status_output:
                print(f"⚠️  Error: {home} vs {away} - {str(e)[:50]}")
    
    # Display results
    display_top_bets()
    display_all_results()
    
    with status_output:
        clear_output()
        value_count = len([r for r in predictions_results if r['has_value']])
        print(f"✅ Processed {len(predictions_results)} matches")
        print(f"   Found {value_count} matches with value bets")
        print(f"   Criteria: Prob ≥ {min_prob:.0%}, EV ≥ {min_ev:.0%}")

def display_top_bets():
    """Display top value bets."""
    with top_bets_output:
        clear_output()
        
        # Get all value matches
        value_matches = [r for r in predictions_results if r['has_value']]
        
        if not value_matches:
            print("=" * 80)
            print("❌ NO VALUE BETS FOUND")
            print("=" * 80)
            print(f"\nCurrent filters:")
            print(f"  • Probability ≥ {min_prob_slider.value:.0%}")
            print(f"  • EV ≥ {min_ev_slider.value:.0%}")
            print(f"\n💡 Try lowering the sliders to see more opportunities")
            return
        
        # Sort by best EV
        value_matches = sorted(value_matches, key=lambda x: x['bets'][0]['ev'] if x['bets'] else 0, reverse=True)
        
        print("=" * 80)
        print("🏆 TOP VALUE BETS (SORTED BY EV)")
        print("=" * 80)
        
        bet_count = 0
        for i, result in enumerate(value_matches[:7], 1):
            if not result['bets']:
                continue
            
            print(f"\n{'=' * 80}")
            print(f"#{i} - {result['match']} ({result['league']})")
            print(f"{'=' * 80}")
            print(f"Expected Goals: {result['xg']:.2f} (H:{result['lambda_home']:.1f} A:{result['lambda_away']:.1f})")
            print(f"")
            
            # Show top 3 bets for this match
            for j, bet in enumerate(result['bets'][:3], 1):
                print(f"🎯 BET #{j}: {bet['market']}")
                print(f"   Probability: {bet['prob']:.1%}")
                print(f"   Odds: {bet['odds']:.2f}")
                print(f"   Edge: {bet['edge']:.2%}")
                print(f"   EV: {bet['ev']:+.2%}")
                print(f"")
                bet_count += 1
        
        print(f"{'=' * 80}")
        print(f"📊 Total value bets shown: {bet_count}")
        print(f"{'=' * 80}")

def display_all_results():
    """Display all match results."""
    with all_results_output:
        clear_output()
        
        if not predictions_results:
            return
        
        print("=" * 80)
        print("📊 ALL MATCHES - COMPLETE RESULTS")
        print("=" * 80)
        
        for i, result in enumerate(predictions_results, 1):
            status = "✅ VALUE" if result['has_value'] else "❌ NO VALUE"
            
            print(f"\n{i}. {result['match']} ({result['league']}) - {status}")
            print(f"   Probabilities: H:{result['prob_home']:.0%} D:{result['prob_draw']:.0%} A:{result['prob_away']:.0%}")
            print(f"   xG: {result['xg']:.1f}")
            
            if result['bets']:
                print(f"   Top bet: {result['bets'][0]['market']} @ {result['bets'][0]['odds']:.2f} (EV: {result['bets'][0]['ev']:+.1%})")
            else:
                print(f"   No value bets found")

def generate_markets(pred, odds, row):
    """Generate all available markets."""
    markets = []
    
    # 1X2
    markets.append(("Home", pred["prob_home"], odds[0]))
    markets.append(("Draw", pred["prob_draw"], odds[1]))
    markets.append(("Away", pred["prob_away"], odds[2]))
    
    # Over/Under
    if "Avg>2.5" in row and not pd.isna(row["Avg>2.5"]):
        markets.append(("Over 2.5", pred["prob_over_25"], row["Avg>2.5"]))
    if "Avg<2.5" in row and not pd.isna(row["Avg<2.5"]):
        markets.append(("Under 2.5", pred["prob_under_25"], row["Avg<2.5"]))
    
    # Asian Handicap
    if "AvgAHH" in row and not pd.isna(row["AvgAHH"]):
        goal_diff = pred["lambda_home"] - pred["lambda_away"]
        prob_home_cover = 1 / (1 + np.exp(-goal_diff))
        prob_away_cover = 1 - prob_home_cover
        markets.append(("Asian Handicap Home", prob_home_cover, row["AvgAHH"]))
        markets.append(("Asian Handicap Away", prob_away_cover, row["AvgAHA"]))
    
    return markets

# Bind buttons
parse_button.on_click(parse_games)
clear_button.on_click(clear_paste)
predict_button.on_click(predict_all)

# ============================================================================
# DISPLAY GUI
# ============================================================================

# Header
display(HTML('''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
            padding: 25px; 
            border-radius: 12px; 
            margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
    <h2 style="color: white; margin: 0; font-size: 28px;">⚽ PROFESSIONAL BETTING MODEL</h2>
    <p style="color: #f0f0f0; margin: 5px 0 0 0; font-size: 14px;">AI-Powered Value Bet Finder</p>
</div>
'''))

# Section 1: Input
input_section = create_styled_box(
    "📋 STEP 1: PASTE GAMES",
    [
        widgets.HTML('''
        <div style="background: #fff3cd; padding: 10px; border-radius: 5px; margin-bottom: 10px; border-left: 4px solid #ffc107;">
            <strong>Format:</strong> League, Home Team, Away Team, Home Odds, Draw Odds, Away Odds<br>
            <strong>Example:</strong> <code>Serie A, Lecce, Cremonese, 2.415, 3.015, 3.705</code>
        </div>
        '''),
        paste_area,
        widgets.HBox([parse_button, clear_button], layout=widgets.Layout(margin='10px 0 0 0')),
        status_output
    ],
    color="#667eea"
)

# Section 2: Controls
controls_section = create_styled_box(
    "⚙️ STEP 2: ADJUST FILTERS",
    [
        widgets.HTML('<p style="margin: 0 0 10px 0; color: #666;">Lower thresholds to see more betting opportunities</p>'),
        min_prob_slider,
        min_ev_slider,
        widgets.HTML('<br>'),
        predict_button
    ],
    color="#764ba2"
)

# Section 3: Top Bets
top_bets_section = create_styled_box(
    "🏆 TOP VALUE BETS",
    [top_bets_output],
    color="#28a745"
)

# Section 4: All Results
all_results_section = create_styled_box(
    "📊 ALL MATCHES",
    [all_results_output],
    color="#17a2b8"
)

# Display all sections
display(input_section)
display(controls_section)
display(top_bets_section)
display(all_results_section)

print("✅ Professional GUI loaded")

✅ Professional GUI loaded
